# M11.9 · Politika gradyanı ve aktör–eleştirmen — REINFORCE, A2C, DPG

Değer tabanlı (Q-öğrenme) ile politika tabanlı (politika gradyanı) yöntemlerin birleşimi. Rastgele politika $\pi_\theta(a|s)$: ayrık eylemlerde softmax, sürekli eylemlerde Gauss. Uzman verisinden en çok olabilirlik (taklit) ve ödülden öğrenme (REINFORCE, $\gamma^nG_n$ terimi), rastgele politika gradyanı teoremi, taban çizgisi ve avantaj, Monte Carlo yerine TD eleştirmen (aktör–eleştirmen), n adımlı avantajla A2C ve sürekli eylemler için deterministik politika gradyanı (DPG). Kısa koridor örneğinde rastgele politikanın optimal olması, NumPy'da teoremin sonlu farklarla doğrulanması, PyTorch'ta paralel ortamlı A2C ile CartPole ve doğrusal– karesel düzenleyicide (LQR) Riccati çözümüne yakınsayan DPG.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m11-9-actor-critic.html](https://m-gonul.github.io/pytorch-lab/m11-9-actor-critic.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m11_9_actor_critic.py
CHECKS = {
 "b": "import math\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def env():\n        f = need(ns, \"corridor_step\")\n        ensure(f(0, True) == (1, -1.0, False) and f(0, False) == (0, -1.0, False), \"S\")\n        ensure(f(1, True) == (0, -1.0, False) and f(1, False) == (2, -1.0, False), \"durum 1 ters\")\n        ensure(f(2, True) == (3, -1.0, True), \"hedef\")\n        v = need(ns, \"corridor_value\")\n        ps = 2 - math.sqrt(2)\n        close(v(ps), (2 * ps - 4) / (ps * (1 - ps)), what=\"J(p*)\")\n        ensure(v(0.5) < v(0.586) > v(0.7), \"p* yerel en büyük\")\n    R_.test(\"ortam ve J(p)\", env)\n\n    def learn():\n        rows, res = [], {}\n        for name, kw in [(\"REINFORCE α = 2⁻¹²\", dict(alpha=2 ** -12)), (\"REINFORCE α = 2⁻¹³\", dict(alpha=2 ** -13)),\n                         (\"taban çizgili α = 2⁻¹⁰, α_w = 2⁻⁶\", dict(alpha=2 ** -10, baseline=True, alpha_w=2 ** -6))]:\n            H = [need(ns, \"reinforce\")(500, seed=s, **kw)[1] for s in range(20)]\n            ensure(all(len(h) == 500 for h in H), \"geçmiş uzunluğu\")\n            J = lambda k: sum(ns[\"corridor_value\"](min(max(h[k], 1e-3), 0.999)) for h in H) / 20\n            res[name] = [J(k) for k in (24, 49, 99, 199, 499)]\n            rows.append([name] + [f\"{v:.1f}\" for v in res[name]])\n        b = res[\"taban çizgili α = 2⁻¹⁰, α_w = 2⁻⁶\"]\n        ensure(b[1] > res[\"REINFORCE α = 2⁻¹²\"][1] + 5 and b[-1] > -13, \"taban çizgisi daha hızlı ve J* = −11.66'ya yakın\")\n        R_.table(\"Kısa koridor: politikanın gerçek değeri J(p), 20 deney ortalaması (J* = −11.66)\", [\"yöntem\", \"25\", \"50\", \"100\", \"200\", \"500. bölüm\"], rows,\n                 note=\"Taban çizgisi beklentiyi değiştirmez, varyansı düşürür; bu yüzden daha büyük α kararlı kalır ve öğrenme hızlanır (Sutton Şekil 13.1–13.2).\")\n    R_.test(\"REINFORCE ve taban çizgisi\", learn)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngü kuralları\", lambda: (max_loops(src, 0, func=\"pg_theorem\"), max_loops(src, 0, func=\"evaluate\"), max_loops(src, 1, func=\"finite_diff\")))\n    rng = np.random.default_rng(5)\n    S, A = 6, 3\n    P = rng.random((A, S, S)) ** 3\n    P /= P.sum(-1, keepdims=True)\n    R = rng.normal(0, 1, (S, A))\n    d0 = rng.dirichlet(np.ones(S))\n\n    def thm():\n        for gamma in (0.5, 0.9, 0.99):\n            th = rng.normal(0, 1, (S, A))\n            close(need(ns, \"pg_theorem\")(th, P, R, d0, gamma), need(ns, \"finite_diff\")(th, P, R, d0, gamma), atol=1e-5, rtol=1e-5, what=f\"teorem = sonlu fark (γ = {gamma})\")\n        pi = ns[\"softmax_policy\"](th)\n        V, Q = ns[\"evaluate\"](P, R, pi, 0.9)\n        close((pi * Q).sum(1), V, atol=1e-10, what=\"V = Σπ Q\")\n    R_.test(\"politika gradyanı teoremi\", thm)\n\n    def ascent():\n        gamma = 0.9\n        th = np.zeros((S, A))\n        rows = []\n        Vs = np.zeros(S)\n        for _ in range(3000):\n            Vs = (R + gamma * np.einsum(\"ast,t->sa\", P, Vs)).max(1)\n        Jstar = float(d0 @ Vs)\n        for it in range(401):\n            if it in (0, 10, 50, 100, 400):\n                rows.append([it, f\"{ns['J'](th, P, R, d0, gamma):.4f}\"])\n            th = th + 5.0 * ns[\"pg_theorem\"](th, P, R, d0, gamma)\n        ensure(Jstar - ns[\"J\"](th, P, R, d0, gamma) < 0.05, \"gradyan yükselişi J*'a yaklaşmalı\")\n        R_.table(f\"Tam gradyanla politika yükselişi (J* = {Jstar:.4f}, değer yinelemesi)\", [\"yineleme\", \"J(θ)\"], rows,\n                 note=\"Softmax parametrelemede optimal deterministik politika ancak θ → ∞'da elde edilir; yaklaşım yavaşlar (gradyan π(1−π) ile söner).\")\n    R_.test(\"gradyan yükselişi\", ascent)\n",
 "d": "import torch\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def nr():\n        r = torch.tensor([[1.0, 1], [1, 1], [1, 1]])\n        d = torch.tensor([[0.0, 0], [0, 1], [0, 0]])\n        out = need(ns, \"nstep_returns\")(r, d, torch.tensor([10.0, 10.0]), 0.5)\n        close(out.numpy(), torch.tensor([[1 + 0.5 + 0.25 + 0.125 * 10, 1 + 0.5], [1 + 0.5 + 0.25 * 10, 1.0], [1 + 0.5 * 10, 1 + 0.5 * 10]]).numpy(), what=\"n adımlı getiriler (bitişte önyükleme yok)\")\n    R_.test(\"nstep_returns\", nr)\n\n    def ac():\n        net = need(ns, \"ActorCritic\")()\n        lo, v = net(torch.zeros(5, 4))\n        ensure(tuple(lo.shape) == (5, 2) and tuple(v.shape) == (5,), \"çıktı şekilleri\")\n    R_.test(\"ActorCritic\", ac)\n\n    def tr():\n        net, lens = need(ns, \"train_a2c\")(1500, 32, 8, 0.99, 2e-3, 0.01, 0)\n        ev = ns[\"evaluate\"](net)\n        first = sum(lens[:50]) / 50\n        ensure(float(ev.mean()) >= 200, f\"A2C CartPole'u öğrenmeli (açgözlü ort. {float(ev.mean()):.0f})\")\n        R_.table(\"A2C, 32 ortam × 8 adım × 1500 güncelleme (384 000 adım)\", [\"ölçü\", \"değer\"],\n                 [[\"ilk 50 bölüm ort. uzunluk\", f\"{first:.0f}\"], [\"son 50 bölüm (rastgele politika) ort.\", f\"{sum(lens[-50:]) / 50:.0f}\"],\n                  [\"açgözlü değerlendirme (20 bölüm, en fazla 500)\", f\"{float(ev.mean()):.0f}\"]],\n                 note=\"Eğitimdeki uzunluklar örneklenen (rastgele) politikadan; açgözlü politika genelde daha iyidir. DQN'den farklı olarak tampon yok: politika üstü, her veri bir kez kullanılır; paralel ortamlar korelasyonu kırar.\")\n    R_.test(\"train_a2c\", tr)\n",
 "e": "import random\nimport torch\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef loop_collect(k, n_envs, T, sigma):\n    rng = random.Random(0)\n    out = []\n    for _ in range(n_envs):\n        s = rng.gauss(0, 1)\n        for _ in range(T):\n            u = k * s + sigma * rng.gauss(0, 1)\n            s2 = 1.1 * s + 0.5 * u + 0.1 * rng.gauss(0, 1)\n            out.append((s, u, -(s * s + 0.5 * u * u), s2))\n            s = max(min(s2, 5), -5)\n    return out\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"yalnız adım döngüsü\", lambda: max_loops(src, 1, func=\"collect\"))\n    P, kstar = ns[\"riccati\"]()\n\n    def q():\n        g = torch.Generator().manual_seed(0)\n        S, U, Rw, S2 = need(ns, \"collect\")(kstar, 2000, 30, 0.5, g)\n        ensure(S.shape == (60000,) and torch.allclose(Rw, -(S ** 2 + 0.5 * U ** 2)), \"geçişler\")\n        w = need(ns, \"lstdq\")(S, U, Rw, S2, kstar)\n        # eylem-değer: Q(s,u) = -(s² + 0.5u²) - γ P (1.1 s + 0.5 u)² + sabit\n        ref = torch.tensor([-(1 + 0.95 * P * 1.21), -(0.95 * P * 2 * 1.1 * 0.5), -(0.5 + 0.95 * P * 0.25)], dtype=torch.float64)\n        close(w[:3].numpy(), ref.numpy(), atol=0.02, what=\"LSTD-Q, optimal politikada Riccati Q fonksiyonuna eşit\")\n    R_.test(\"collect ve lstdq\", q)\n\n    def d():\n        rows = []\n        for name, a in [(\"tam iyileştirme (k ← −w₁/2w₂)\", None), (\"DPG α = 0.3\", 0.3), (\"DPG α = 0.1\", 0.1)]:\n            k, h = need(ns, \"dpg\")(40, -0.5, a, 256, 20, 0.5, 1)\n            rows.append([name] + [f\"{h[i]:.3f}\" for i in (0, 1, 5, 20, 40)])\n            if a != 0.1:\n                ensure(abs(k - kstar) < 0.03, f\"{name}: k* = {kstar:.3f}'e yakınsamalı (bulunan {k:.3f})\")\n        t_vec = bench(lambda: ns[\"collect\"](-1.0, 256, 20, 0.5, torch.Generator().manual_seed(0)), repeat=3, number=1)\n        t_loop = bench(lambda: loop_collect(-1.0, 256, 20, 0.5), repeat=3, number=1)\n        rows.append([\"süre: 256 ortam toplu toplama\", fmt_time(t_vec), \"\", \"\", \"\", \"\"])\n        rows.append([\"süre: ortam döngüsü\", fmt_time(t_loop), \"\", \"\", \"\", \"\"])\n        R_.table(f\"LQR, k* = {kstar:.4f} (Riccati)\", [\"yöntem\", \"k₀\", \"k₁\", \"k₅\", \"k₂₀\", \"k₄₀\"], rows,\n                 note=\"Q bu problemde tam olarak karesel olduğundan doğrusal eleştirmen yanlılıksızdır; ∂Q/∂u = w₁s + 2w₂u eleştirmenin aktöre verdiği yön. Tam iyileştirme politika yinelemesidir (birkaç adımda yakınsar); gradyan adımları DDPG'nin aktörüne karşılık gelir.\")\n    R_.test(\"dpg\", d)\n",
 "x1": "from labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        mus, sig = need(ns, \"gauss_reinforce\")(400, 64, 0.05, 1)\n        ensure(abs(mus[-1] - 3) < 0.15 and sig[-1] < 0.5 * sig[0], f\"μ → 3 ve σ küçülmeli ({mus[-1]:.2f}, {sig[-1]:.2f})\")\n        R_.table(\"Gauss politikası, ödül −(a − 3)²\", [\"yineleme\", \"μ\", \"σ\"], [[k, f\"{mus[k]:.3f}\", f\"{sig[k]:.3f}\"] for k in (0, 25, 50, 100, 200, 400)],\n                 note=\"σ keşif miktarıdır ve kendiliğinden azalır: optimuma yaklaştıkça geniş σ ortalama ödülü düşürür. Doğal gradyan (σ² ile ölçekleme) adımları ölçek-bağımsız kılar.\")\n    R_.test(\"Gauss REINFORCE\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rows = []\n        res = {}\n        for n in (50, 200, 1000, 5000):\n            a = np.mean([need(ns, \"bc\")(n, 0.5, s) for s in range(5)])\n            b = np.mean([need(ns, \"pg\")(n, 0.5, s) for s in range(5)])\n            res[n] = (a, b)\n            rows.append([n, f\"{a:.3f}\", f\"{b:.3f}\"])\n        ensure(res[200][0] > res[200][1] and res[5000][1] > 0.85, \"taklit daha hızlı; REINFORCE de sonunda öğrenir\")\n        R_.table(\"Doğrusal softmax politika, test doğruluğu (5 tohum)\", [\"etkileşim\", \"taklit (uzman etiketi)\", \"REINFORCE (0/1 ödül)\"], rows,\n                 note=\"Uzman etiketi her örnekte doğru eylemi söyler (log K bit); ödül yalnız seçilen eylemin doğru olup olmadığını söyler (≤ 1 bit) ve önce keşif ister. AlphaGo önce uzman oyunlarıyla taklit, sonra kendine karşı REINFORCE kullanır.\")\n    R_.test(\"taklit vs politika gradyanı\", t)\n",
 "x3": "import math\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        tg = need(ns, \"true_grad\")()\n        rows = []\n        var = {}\n        for kind, name in [(\"G\", \"taban çizgisiz\"), (\"b\", \"sabit b = J(p)\"), (\"V\", \"durum taban çizgisi V(s)\")]:\n            xs = need(ns, \"grad_samples\")(20000, kind, 1)\n            m = sum(xs) / len(xs)\n            v = sum((x - m) ** 2 for x in xs) / (len(xs) - 1)\n            se = math.sqrt(v / len(xs))\n            ensure(abs(m - tg) < 4 * se, f\"{name}: yansız olmalı ({m:.2f} vs {tg:.2f})\")\n            var[kind] = v\n            rows.append([name, f\"{m:.3f}\", f\"{se:.3f}\", f\"{v:.1f}\"])\n        ensure(var[\"V\"] < var[\"G\"], \"durum taban çizgisi varyansı düşürmeli\")\n        R_.table(f\"∂J/∂θ₀ kestiricileri, p = 0.3, 20 000 bölüm (gerçek: {tg:.3f})\", [\"kestirici\", \"ortalama\", \"std. hata\", \"varyans\"], rows,\n                 note=\"Üçü de yansız; varyans farkı aynı doğruluk için gereken bölüm sayısının oranıdır.\")\n    R_.test(\"varyans\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m11_9_actor_critic as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Üç aile:** Q-öğrenme (modelsiz, değer tabanlı; açık politika yok), politika gradyanı (modelsiz, politika tabanlı; açık değer yok), **aktör–eleştirmen** (ikisi birden).

**Rastgele politika** $\pi_\theta(a|s)=P(a|s;\theta)$: durumdan bir eylem değil, eylemler üzerinde dağılım verir. Ayrık: $\mathrm{softmax}(h_\theta(s,\cdot))$; sürekli: $N(a\,|\,\mu_\theta(s),\sigma_\theta(s)^2)$.

- **Taklit:** uzman çiftleri $(s_i,a_i^*)$ → $\max_\theta\sum_i\log\pi_\theta(a_i^*|s_i)$ (AlphaGo'nun ilk aşaması).
- **REINFORCE:** $\nabla_\theta\leftarrow\sum_n\gamma^nG_n\nabla\log\pi_\theta(a_n|s_n)$, $\theta\leftarrow\theta+\alpha\nabla_\theta$. Toplam indirimli getirisi büyük eylemlerin olasılığı artar.

**Politika gradyanı teoremi** (Sutton ve ark. 2000): $\nabla J(\theta)=\sum_s\mu_\theta(s)\sum_a\nabla\pi_\theta(a|s)\,Q_\theta(s,a)$; $\mu_\theta$ (indirimli) durum dağılımı. Durum dağılımının gradyanı gerekmez — kanıtın zor ve şaşırtıcı kısmı budur. Tek örnekli kestirim REINFORCE'u verir.

**Taban çizgisi:** $\sum_a\nabla\pi(a|s)\,(Q(s,a)-b(s))$ aynı gradyan, çünkü $\sum_a\nabla\pi=\nabla1=0$. $b(s)\approx V^\pi(s)$ → **avantaj** $A(s,a)=Q(s,a)-V(s)$: "bu durumda bu eylem ortalamadan ne kadar iyi". Politika ağı θ, değer ağı w.

- REINFORCE + taban çizgisi: $V_w$'yi MC ile öğren, $\delta=G_n-V_w(s_n)$.
- **Aktör–eleştirmen:** $V_w$'yi TD ile öğren, $\delta=r+\gamma V_w(s')-V_w(s)$ — düşük varyans, hızlı yakınsama (biraz yanlılık).
- **A2C:** n adımlı avantaj, paralel ortamlar, entropi bonusu.

**DPG** (Silver ve ark. 2014): sürekli eylem, deterministik $\mu_\theta(s)$: $\nabla J=\mathbb E_s[\nabla_\theta\mu_\theta(s)\nabla_aQ(s,a)|_{a=\mu_\theta(s)}]$. Keşif davranış gürültüsüyle (politika dışı), eleştirmen $Q_w$.

## 1 · Elle

<!-- çözüm -->

Softmax: $\partial\log\pi(a)/\partial\theta_b=\mathbb 1[a=b]-\pi(b)$ → $1-1/3$. Gauss: $\partial\log N(a|\mu,\sigma^2)/\partial\mu=(a-\mu)/\sigma^2$.

DPG: $\nabla_aQ=-2(a-2)=3$ ($a=\mu=0.5$), $\nabla_\theta\mu=s=1$ → 3: θ büyüyecek, eylem 2'ye doğru.

Koridor: $dJ/dp=0$ → $p^2-4p+2=0$ → $p^*=2-\sqrt2$.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Log-olabilirlik gradyanları**

- `sm`: Softmax, θ = (0,0,0), örneklenen eylem 0: $\partial\log\pi(0)/\partial\theta_0$
- `ga`: Gauss, σ = 1, μ = 1, a = 1.5: $\partial\log\pi/\partial\mu$

**Güncelleme katsayıları**

- `rf`: REINFORCE: n = 2, γ = 0.9, $G_n=10$: $\gamma^nG_n$
- `ad`: Aktör–eleştirmen: r = 1, γ = 0.9, V(s′) = 5, V(s) = 4: δ
- `dp`: DPG: $Q(s,a)=-(a-2)^2$, $\mu_\theta(s)=\theta s$, s = 1, θ = 0.5: $\nabla_\theta J$

**Kısa koridor (Sutton Örnek 13.1): $J(p)=\dfrac{2p-4}{p(1-p)}$**

- `ps`: En iyi 'sağ' olasılığı $p^*$
- `js`: $J(p^*)$

In [ ]:
cevaplar = dict(
    sm=None,
    ga=None,
    rf=None,
    ad=None,
    dp=None,
    ps=None,
    js=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"sm":[0.6666666666666666,1e-9],"ga":[0.5,0],"rf":[8.1,0],"ad":[1.5,0],"dp":[3,0],"ps":[0.5857864376269049,0.000001],"js":[-11.65685424949238,0.000001]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

**Kısa koridor** (Sutton Örnek 13.1), saf Python. Durumlar 0 (S), 1, 2, 3 (hedef). Eylem sağ/sol; **durum 1'de eylemler ters çalışır**. Solda duvar. Her adım −1, γ = 1. Tüm durumların özniteliği aynı: $x(s,\text{sağ})=(1,0)$, $x(s,\text{sol})=(0,1)$, $\pi(\text{sağ})=\sigma(\theta_0-\theta_1)$.

- `corridor_step(s, right)` → `(s2, r, done)`.
- `corridor_value(p)` — $J(p)=(2p-4)/(p(1-p))$.
- `reinforce(n_eps, alpha, seed=0, baseline=False, alpha_w=0.0, max_steps=1000)` — θ başlangıcı $\pi(\text{sağ})=0.05$; her bölümden sonra her adım için (γ = 1) $\delta=G_t-w$ (taban çizgisi yoksa $\delta=G_t$), önce $w\leftarrow w+\alpha_w\delta$, sonra $\theta\leftarrow\theta+\alpha\,\delta\,\nabla\log\pi$. Çıktı: `(theta, her bölüm sonrası p listesi)`.

In [ ]:
%%writefile stage_b.py
import math
import random


def corridor_step(s, right):
    ...


def corridor_value(p):
    ...


def reinforce(n_eps, alpha, seed=0, baseline=False, alpha_w=0.0, max_steps=1000):
    ...


theta, ps = reinforce(200, 2 ** -10, baseline=True, alpha_w=2 ** -6)
print(ps[-1], corridor_value(ps[-1]))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

**Teoremi doğrula**, NumPy. Tablo softmax politika $\pi(a|s)\propto e^{\theta_{s,a}}$, `P (A, S, S)`, `R (S, A)`, başlangıç dağılımı `d0`.

- `softmax_policy(theta)` → `(S, A)`.
- `evaluate(P, R, pi, gamma)` → `(V, Q)`: $V=(I-\gamma P_\pi)^{-1}r_\pi$, $Q=R+\gamma\,P V$.
- `J(theta, P, R, d0, gamma)` = $d_0^\top V$.
- `pg_theorem(theta, P, R, d0, gamma)` → `(S, A)`: $d_\gamma=(I-\gamma P_\pi^\top)^{-1}d_0$ (indirimli ziyaret), $\partial J/\partial\theta_{s,b}=d_\gamma(s)\,\pi(b|s)\,(Q(s,b)-V(s))$.
- `finite_diff(theta, P, R, d0, gamma, eps=1e-6)` — merkezi farklar (tek döngü serbest).

In [ ]:
%%writefile stage_c.py
import numpy as np


def softmax_policy(theta):
    ...


def evaluate(P, R, pi, gamma):
    ...


def J(theta, P, R, d0, gamma):
    ...


def pg_theorem(theta, P, R, d0, gamma):
    ...


def finite_diff(theta, P, R, d0, gamma, eps=1e-6):
    ...


rng = np.random.default_rng(0)
P = rng.random((2, 4, 4)); P /= P.sum(-1, keepdims=True)
R = rng.random((4, 2)); d0 = np.full(4, 0.25); th = rng.normal(0, 1, (4, 2))
print(np.abs(pg_theorem(th, P, R, d0, 0.9) - finite_diff(th, P, R, d0, 0.9)).max())

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

**A2C**, CartPole (fizik verildi), 32 paralel ortam.

- `class ActorCritic(nn.Module)` — gövde 4→64→64 (tanh), başlıklar: politika logit'leri (2), değer (1). `forward(x)` → `(logits, v)`.
- `nstep_returns(r, d, v_last, gamma)` — `r, d (T, N)`; $R_t=r_t+\gamma(1-d_t)R_{t+1}$, $R_T=V(s_T)$.
- `train_a2c(updates=1500, n_envs=32, n_steps=8, gamma=0.99, lr=2e-3, ent=0.01, seed=0)` → `(ağ, bölüm uzunlukları)`: her güncellemede n adım topla (Categorical örnekleme), kayıp $-\overline{\log\pi\cdot A}+0.5\,\overline{A^2}-\text{ent}\cdot\overline{H}$, $A=R-V$ (aktörde `detach`), gradyan normu 0.5'te kırpılır. 500 adımda kesme; biten ortam yeniden başlar.

In [ ]:
%%writefile stage_d.py
import math
import torch
import torch.nn as nn

G_, MC, MP, L, FM, TAU = 9.8, 1.0, 0.1, 0.5, 10.0, 0.02


def cartpole_step(x, a):
    pos, vel, th, om = x.unbind(1)
    f = torch.where(a == 1, FM, -FM).to(x.dtype)
    ct, st = torch.cos(th), torch.sin(th)
    tm, pml = MC + MP, MP * L
    temp = (f + pml * om ** 2 * st) / tm
    thacc = (G_ * st - ct * temp) / (L * (4.0 / 3.0 - MP * ct ** 2 / tm))
    xacc = temp - pml * thacc * ct / tm
    x2 = torch.stack([pos + TAU * vel, vel + TAU * xacc, th + TAU * om, om + TAU * thacc], 1)
    done = (x2[:, 0].abs() > 2.4) | (x2[:, 2].abs() > 12 * 2 * math.pi / 360)
    return x2, done


def evaluate(net, n=20, max_len=500, seed=123):
    g = torch.Generator().manual_seed(seed)
    x = (torch.rand(n, 4, generator=g) - 0.5) * 0.1
    alive = torch.ones(n, dtype=torch.bool)
    lens = torch.zeros(n)
    for _ in range(max_len):
        with torch.no_grad():
            x, d = cartpole_step(x, net(x)[0].argmax(1))
        lens += alive.float()
        alive &= ~d
    return lens


class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        ...

    def forward(self, x):
        ...


def nstep_returns(r, d, v_last, gamma):
    ...


def train_a2c(updates=1500, n_envs=32, n_steps=8, gamma=0.99, lr=2e-3, ent=0.01, seed=0):
    ...


net, lens = train_a2c(100)
print(len(lens), evaluate(net).mean())

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

**DPG ile LQR**, PyTorch, paralel ortamlar. $s'=1.1s+0.5u+0.1\epsilon$, ödül $-(s^2+0.5u^2)$, γ = 0.95. Politika $u=ks$ (deterministik), davranış $u=ks+\sigma\epsilon$.

- `riccati()` → `(P, k*)` — Riccati yinelemesi (verildi).
- `collect(k, n_envs, T, sigma, g)` → `(S, U, R, S2)` düzleştirilmiş; adım döngüsü serbest, ortam döngüsü yok; $s$ her adım $[-5,5]$'e kırpılır.
- `lstdq(S, U, R, S2, k)` — karesel öznitelikler $\phi=(s^2,su,u^2,1)$, LSTD-Q: $w=\big[\Phi^\top(\Phi-\gamma\Phi')\big]^{-1}\Phi^\top r$, $\Phi'=\phi(s',ks')$.
- `dpg(n_iter=40, k0=-0.5, alpha=None, n_envs=256, T=20, sigma=0.5, seed=0)` → `(k, geçmiş)`: $\nabla_kJ\approx\overline{s^2}\,(w_1+2w_2k)$ (çünkü $\partial Q/\partial u=w_1s+2w_2u$, $\partial\mu/\partial k=s$); `alpha=None` ise tam iyileştirme $k\leftarrow-w_1/(2w_2)$ (politika yinelemesi).

In [ ]:
%%writefile stage_e.py
import torch

A_, B_, Q_, R_, GAM = 1.1, 0.5, 1.0, 0.5, 0.95


def riccati(iters=2000):
    P = 0.0
    for _ in range(iters):
        P = Q_ + GAM * A_ * A_ * P - (GAM * A_ * B_ * P) ** 2 / (R_ + GAM * B_ * B_ * P)
    return P, -GAM * A_ * B_ * P / (R_ + GAM * B_ * B_ * P)


def collect(k, n_envs, T, sigma, g):
    ...


def lstdq(S, U, R, S2, k):
    ...


def dpg(n_iter=40, k0=-0.5, alpha=None, n_envs=256, T=20, sigma=0.5, seed=0):
    ...


print(riccati()[1], dpg()[0])

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Gauss politikası: sürekli eylemde REINFORCE

**Gauss politikası:** sürekli tek adımlı problem, ödül $-(a-3)^2+\text{gürültü}$. Parametreler μ ve $\log\sigma$.

- `gauss_reinforce(n_iter, batch=64, lr=0.05, seed=0)` → `(mu, sigma)` geçmişi: her yinelemede batch kadar $a\sim N(\mu,\sigma^2)$, taban çizgisi = batch ortalama ödülü; $\nabla_\mu\log\pi=(a-\mu)/\sigma^2$, $\nabla_{\log\sigma}\log\pi=(a-\mu)^2/\sigma^2-1$. μ adımı σ² ile ölçeklenir (doğal gradyan: $\mu\mathrel{+}=\text{lr}\cdot\overline{A\,(a-\mu)}$), $\log\sigma$ adımı lr/5. NumPy, yineleme döngüsü serbest.

In [ ]:
%%writefile extra_x1.py
import numpy as np


def gauss_reinforce(n_iter, batch=64, lr=0.05, seed=0):
    ...


mus, sigmas = gauss_reinforce(200)
print(mus[-1], sigmas[-1])

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Taklit (en çok olabilirlik) mi, ödülden öğrenme mi?

Bağlam $x\in\mathbb R^2$, 3 eylem, uzman $a^*=\arg\max_kW^{*\top}_kx$. Doğrusal softmax politika.

- `bc(n_samples, lr=0.5, seed=0)` — uzman etiketleriyle çapraz entropi (her örnekte bir SGD adımı, NumPy).
- `pg(n_samples, lr=0.5, seed=0)` — aynı sayıda etkileşim; eylem örnekle, ödül $\mathbb 1[a=a^*]$, taban çizgisi koşan ortalama; REINFORCE.
- Her ikisi: test doğruluğu (1000 bağlam, açgözlü).

In [ ]:
%%writefile extra_x2.py
import numpy as np


def bc(n_samples, lr=0.5, seed=0):
    ...


def pg(n_samples, lr=0.5, seed=0):
    ...


print(bc(500), pg(500))

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Gradyan kestiricilerinin varyansı

Koridorda sabit θ ($p=0.3$) için $\partial J/\partial\theta_0$ kestiricileri (tek bölüm başına):

- `grad_samples(n, kind, seed=0)` → n tahmin (liste); `kind`: `"G"` ($\sum_tG_t\nabla\log\pi$), `"b"` (sabit taban çizgisi $J(p)$), `"V"` (durum değeri $V(s_t)$, gerçek değerler).
- `true_grad()` — $dJ/d\theta_0=J'(p)\,p(1-p)$.

In [ ]:
%%writefile extra_x3.py
import math
import random


def true_grad():
    ...


def grad_samples(n, kind, seed=0):
    ...


print(true_grad(), sum(grad_samples(2000, "V")) / 2000)

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Silver ve ark. (2014): Deterministik politika gradyanı

[Silver ve ark. — Deterministic Policy Gradient Algorithms (ICML 2014)](https://proceedings.mlr.press/v32/silver14.html)

Teorem 1'i (deterministik politika gradyanı) ve rastgele politika gradyanının σ → 0 limiti olduğunu (Teorem 2) LQR üzerinde sayısal olarak göster. COPDAC-Q'yu (uyumlu doğrusal eleştirmen, Bölüm 4.3) uygula ve e aşamasındaki LQR'de, ardından d-boyutlu LQR'de (d = 10, 50) rastgele politika gradyanıyla (aynı örnek bütçesi) karşılaştır.


**Adımlar**

1. Uyumlu öznitelikler: $\phi(s,a)=(a-\mu_\theta(s))^\top\nabla_\theta\mu_\theta(s)$.
2. Gradyanın varyansını σ'nın fonksiyonu olarak ölç.

**Kabul ölçütleri**

- Eylem boyutu arttıkça DPG'nin üstünlüğü büyür (makale Şekil 1'in nitel biçimi).
- σ → 0'da rastgele politika gradyanının varyansı patlar, DPG'ninki sonlu kalır.

## Makale ödevi: Mnih ve ark. (2016): A3C / A2C

[Mnih ve ark. — Asynchronous Methods for Deep Reinforcement Learning (ICML 2016)](https://arxiv.org/abs/1602.01783)

d aşamasındaki A2C'yi (eşzamanlı) Algoritma S3'teki asenkron sürüme (A3C, `torch.multiprocessing` ile Hogwild paylaşımlı parametreler) dönüştür; CartPole ve LunarLander'da işçi sayısının (1, 2, 4, 8) duvar saati süresi ve örnek verimliliği üzerindeki etkisini ölç. Entropi düzenlemesinin etkisini ablation ile göster.


**Adımlar**

1. Paylaşımlı RMSProp istatistikleri (makale Bölüm 4).

**Kabul ölçütleri**

- İşçi sayısıyla duvar saati hızlanması ≈ doğrusal (makale Şekil 3 nitel).
- Entropi bonusu olmadan erken deterministikleşme ve daha kötü son performans.

## Kaynaklar

- [Sutton, McAllester, Singh, Mansour — Policy Gradient Methods for RL with Function Approximation (NeurIPS 2000)](https://papers.nips.cc/paper/1999/hash/464d828b85b0bed98e80ade0a5c43b0f-Abstract.html) — ©
- [Mnih ve ark. — Asynchronous Methods for Deep Reinforcement Learning (ICML 2016)](https://arxiv.org/abs/1602.01783) — arXiv
- [Silver ve ark. — Deterministic Policy Gradient Algorithms (ICML 2014)](https://proceedings.mlr.press/v32/silver14.html) — PMLR
- [Lillicrap ve ark. — Continuous control with deep reinforcement learning (DDPG, ICLR 2016)](https://arxiv.org/abs/1509.02971) — arXiv
- [Sutton & Barto — Reinforcement Learning: An Introduction (2. baskı), Bölüm 13](http://incompleteideas.net/book/the-book-2nd.html) — CC BY-NC-ND 2.0
- [Gymnasium — CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/) — MIT